# 🎬 4DGaussians-Enhanced v1.1: Hybrid Pipeline (Poses + Geometry)

**REVISION:** Sequential pipeline combining:
- **Pose Accuracy** from Calibration Clip (Section 3.2)
- **Geometric Detail** from Extraview Strategy (Section 3.5)

**Pipeline Flow:** Setup → Calib (3.2) → Extraview Recon (3.5) → Alignment/Merge (3.6) → Training


## 📦 Cell 1: Kurulum (Installation)

Tüm bağımlılıkları kurar: 4DGaussians, SAM2, COLMAP, C++ submodule'ler

In [ ]:
# ============================================================
# CELL 1: INSTALLATION (getcwd HATASI DÜZELTİLDİ)
# ============================================================
import os
import sys
import shutil

PROJECT_DIR = "/content/4DGaussians-Enhanced"

# ==========================================
# 🚨 KRİTİK DÜZELTME: GÜVENLİ BÖLGEYE ÇIK
# ==========================================
# Eğer zaten projenin içindeysek, silmeden önce dışarı çıkmalıyız.
# Yoksa "getcwd: cannot access parent directories" hatası alırız.
os.chdir("/content")
print(f"📍 Güvenli ana dizine geçildi: {os.getcwd()}")

# 1. Temizlik (Temiz bir başlangıç için)
if os.path.exists(PROJECT_DIR):
    print(f"🧹 Eski kurulum temizleniyor: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

# 2. Repo'yu Klonla
print("\n📥 Repo klonlanıyor (Loglar açık)...")
!git clone https://github.com/semhfe/4DGaussians-Enhanced.git

# 3. Proje Klasörüne Gir
os.chdir(PROJECT_DIR)
print(f"📂 Proje dizinine girildi: {os.getcwd()}")

# 4. Doğru Branch'e Geç
print("\n🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...")
!git checkout copilot/fix-4dgaussians-enhanced-errors

# 5. Alt Modülleri İndir (KRİTİK ADIM)
print("\n📦 Alt modüller (Submodules) indiriliyor...")
!git submodule update --init --recursive

# 6. requirements.txt Düzenleme
print("\n🔧 'requirements.txt' düzenleniyor (Torch/MMCV temizliği)...")
!sed -i '/torch/d' requirements.txt
!sed -i '/mmcv/d' requirements.txt
!cat requirements.txt | head -n 5

# 7. Bağımlılıkları Yükleme (LOGLAR AÇIK)
print("\n📦 Python kütüphaneleri kuruluyor (Detaylı çıktı)...")
!pip install matplotlib lpips plyfile pytorch_msssim open3d imageio[ffmpeg] opencv-python
!pip install ultralytics supervision huggingface_hub
# SAM2'yi kaynaktan kur
!pip install "git+https://github.com/facebookresearch/sam2.git"

# 8. Setup Scriptini Çalıştırma
print("\n🔧 Setup scripti çalıştırılıyor (C++ Yamaları, Ninja & COLMAP)...")
if os.path.exists("scripts/colab_setup.py"):
    !python scripts/colab_setup.py
else:
    print("❌ HATA: 'scripts/colab_setup.py' dosyası bulunamadı!")
    print("   Lütfen branch isminin doğru olduğundan emin olun.")
    # Dosya yapısını kontrol et
    if os.path.exists("scripts"):
        print(f"   Mevcut dosyalar: {os.listdir('scripts')}")
    else:
        print("   'scripts' klasörü bile yok! Klonlama hatalı olabilir.")

print("\n" + "="*50)
print("✅ Kurulum tamamlandı! (Lütfen yukarıdaki loglarda 'error' olup olmadığını kontrol edin)")
print("="*50)

📍 Güvenli ana dizine geçildi: /content

📥 Repo klonlanıyor (Loglar açık)...
Cloning into '4DGaussians-Enhanced'...
remote: Enumerating objects: 2706, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 2706 (delta 39), reused 50 (delta 21), pack-reused 2625 (from 1)
Receiving objects: 100% (2706/2706), 66.49 MiB | 48.08 MiB/s, done.
Resolving deltas: 100% (1255/1255), done.
📂 Proje dizinine girildi: /content/4DGaussians-Enhanced

🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...
Branch 'copilot/fix-4dgaussians-enhanced-errors' set up to track remote branch 'copilot/fix-4dgaussians-enhanced-errors' from 'origin'.
Switched to a new branch 'copilot/fix-4dgaussians-enhanced-errors'

📦 Alt modüller (Submodules) indiriliyor...
Submodule 'submodules/depth-diff-gaussian-rasterization' (https://github.com/ingra14m/depth-diff-gaussian-rasterization) registered for path 'submodules/depth-diff-gaussian-rasterization'


In [ ]:
# ============================================================
# CELL 1.5: FINAL COMPILATION (C++ Modüllerini Derle)
# ============================================================
import os
import sys

# Proje dizininde olduğumuzdan emin olalım
os.chdir("/content/4DGaussians-Enhanced")

print("🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...")

# 1. Rasterizer Derleme
print("\n📦 Compiling Diff-Gaussian-Rasterization...")
!pip install -e submodules/depth-diff-gaussian-rasterization

# 2. KNN Derleme
print("\n📦 Compiling Simple-KNN...")
!pip install -e submodules/simple-knn

print("\n✅ Derleme tamamlandı! Artık Cell 2'ye geçebilirsiniz.")

🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...

📦 Compiling Diff-Gaussian-Rasterization...
Obtaining file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of diff-gaussian-rasterization==0.0.0 from file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for diff-gaussian-rasterization

📦 Compiling Simple-KNN...
Obtaining file:///content/4DGaussians-Enhanced/submodules/simple-knn
  Preparing m

## 📁 Cell 2: Veri Hazırlama (Data Setup)

Google Drive'ı mount eder, veriyi unzip eder ve formatı doğrular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

print("="*60)
print("📁 Veri Hazırlama")
print("="*60)

# Step 1: Mount Google Drive
print("\n📂 Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("✅ Drive mount edildi")

# Step 2: Configure paths
# BURADAN DÜZENLEYIN: Veri yollarınızı belirtin
DATA_SOURCE = "/content/drive/MyDrive/4DGS_project/input/added_environment.zip"  # Zip dosyası veya klasör yolu
OUTPUT_BASE = "/content/drive/MyDrive/4DGS_project/output"  # Çıktıların kaydedileceği Drive klasörü

# Local processing paths (faster than Drive)
LOCAL_DATA = "/content/data/my_scene"  # Lokal veri klasörü (işleme için)
LOCAL_OUTPUT = "/content/output"  # Lokal çıktı (eğitim için)

# Step 3: Extract or copy data to local disk
os.makedirs(LOCAL_DATA, exist_ok=True)

if DATA_SOURCE.endswith('.zip'):
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Zip dosyası bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📦 Zip açılıyor: {DATA_SOURCE}")
        with zipfile.ZipFile(DATA_SOURCE, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATA)
        print(f"✅ Zip açıldı: {LOCAL_DATA}")
else:
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Klasör bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📂 Veri kopyalanıyor: {DATA_SOURCE} -> {LOCAL_DATA}")
        if os.path.exists(LOCAL_DATA):
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(DATA_SOURCE, LOCAL_DATA)
        print(f"✅ Veri kopyalandı")

# Step 4: Detect data format
print("\n🔍 Veri formatı algılanıyor...")
contents = os.listdir(LOCAL_DATA)
print(f"   İçerik: {contents}")

data_format = None
if 'transforms_train.json' in contents:
    data_format = 'blender'
    print("✅ Format: Blender/NeRF Synthetic")
elif 'sparse' in contents or 'images' in contents:
    data_format = 'colmap'
    print("✅ Format: COLMAP")
elif any('cam' in item for item in contents):
    data_format = 'multicam'
    print("✅ Format: Multi-camera (cam01, cam02, ...)")
elif len([f for f in contents if f.endswith(('.jpg', '.png'))]) > 0:
    data_format = 'raw_images'
    print("✅ Format: Ham resimler (COLMAP gerekli)")
else:
    print("⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.")

# Step 5: Create output directory
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("\n" + "="*60)
print("✅ Veri hazırlama tamamlandı!")
print("="*60)
print(f"\n📁 Lokal veri: {LOCAL_DATA}")
print(f"📁 Lokal çıktı: {LOCAL_OUTPUT}")
print(f"📁 Drive çıktı: {OUTPUT_BASE}")
print(f"\n📊 Format: {data_format}")

if data_format == 'raw_images':
    print("\n⚠️  Ham resimler tespit edildi!")
    print("   Cell 3'ü çalıştırarak COLMAP ile kamera pozlarını hesaplayın")
else:
    print("\n📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")

📁 Veri Hazırlama

📂 Google Drive mount ediliyor...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mount edildi

📦 Zip açılıyor: /content/drive/MyDrive/4DGS_project/input/added_environment.zip
✅ Zip açıldı: /content/data/my_scene

🔍 Veri formatı algılanıyor...
   İçerik: ['added_environment', '__MACOSX']
⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.

✅ Veri hazırlama tamamlandı!

📁 Lokal veri: /content/data/my_scene
📁 Lokal çıktı: /content/output
📁 Drive çıktı: /content/drive/MyDrive/4DGS_project/output

📊 Format: None

📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun


In [ ]:
# ============================================================
# CELL 2.5: COLMAP INSTALLATION (ÖN HAZIRLIK)
# ============================================================
# Bu hücreyi Cell 3'ten ÖNCE çalıştırın.
# Sistemde COLMAP yüklü değilse otomatik olarak kurar.
# ============================================================

import shutil
import os

print("🔍 COLMAP kurulumu kontrol ediliyor...")

# COLMAP komutu sistemde var mı bak
if not shutil.which("colmap"):
    print("📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...")
    try:
        # 1. Paket listesini güncelle (Sessiz mod)
        !apt-get update
        # 2. COLMAP'i kur (Sessiz mod, onay istemeden)
        !apt-get install -y colmap
        print("✅ COLMAP başarıyla kuruldu!")
    except Exception as e:
        print(f"❌ Kurulum sırasında hata oluştu: {e}")
        print("👉 İpucu: '!apt-get install -y colmap' komutunu manuel deneyebilirsiniz.")
else:
    print("✅ COLMAP zaten sistemde yüklü, kuruluma gerek yok.")

# Kurulumu doğrula
print("-" * 30)
print("Sürüm Kontrolü:")
!colmap help | head -n 1

🔍 COLMAP kurulumu kontrol ediliyor...
📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illi

## 📷 Cell 3.2: Phase 1 - Rig Calibration (Perfect Poses)

**Goal:** Extract camera poses from the calibration clip (rig).

**Output:** `/content/output/sparse_calibrated`


In [ ]:
# ============================================================
# CELL 3.2: PHASE 1 - RIG CALIBRATION (PERFECT POSES)
# ============================================================
# Goal: Extract camera poses from a calibration rig clip
# Output: /content/output/sparse_calibrated
# ============================================================

import os
import shutil
import subprocess
import re

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Path to calibration images (8 camera folders: cam01, cam02, ... cam08)
# OR 8 video files that will be extracted
CALIBRATION_INPUT = "/content/drive/MyDrive/4DGS_project/input/calibration_clip"

# Output paths
CALIB_WORKSPACE = "/content/colmap_rig_workspace"
SPARSE_CALIBRATED = "/content/output/sparse_calibrated"

# ==============================================================================
# HYBRID INPUT DETECTION (Images OR Videos)
# ==============================================================================
print("=" * 60)
print("🎯 PHASE 1: RIG CALIBRATION - PERFECT POSES")
print("=" * 60)

def detect_input_type(input_path):
    """Detect if input contains image folders or video files."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ Input path not found: {input_path}")
    
    contents = os.listdir(input_path)
    
    # Priority 1: Check for camera subfolders (cam01, cam02, etc.)
    cam_folders = sorted([f for f in contents if os.path.isdir(os.path.join(input_path, f)) 
                          and f.lower().startswith('cam')])
    if len(cam_folders) >= 8:
        print(f"✅ Detected {len(cam_folders)} camera folders (image mode)")
        return "images", cam_folders[:8]
    
    # Priority 2: Check for video files
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    video_files = sorted([f for f in contents if f.lower().endswith(video_extensions)])
    if len(video_files) >= 8:
        print(f"✅ Detected {len(video_files)} video files (video mode)")
        return "videos", video_files[:8]
    
    raise ValueError(f"❌ Could not detect 8 cameras. Found: {contents}")

input_type, inputs = detect_input_type(CALIBRATION_INPUT)

# ==============================================================================
# PREPARE WORKSPACE
# ==============================================================================
print("\n📁 Preparing workspace...")

# Clean workspace
if os.path.exists(CALIB_WORKSPACE):
    shutil.rmtree(CALIB_WORKSPACE)
os.makedirs(CALIB_WORKSPACE, exist_ok=True)

RIG_IMAGES_DIR = os.path.join(CALIB_WORKSPACE, "images", "rig1")
os.makedirs(RIG_IMAGES_DIR, exist_ok=True)

# Map inputs to cam01...cam08
if input_type == "images":
    # Copy/symlink image folders
    for i, folder in enumerate(inputs):
        src = os.path.join(CALIBRATION_INPUT, folder)
        dst = os.path.join(RIG_IMAGES_DIR, f"cam{i+1:02d}")
        shutil.copytree(src, dst)
        print(f"   📂 {folder} → cam{i+1:02d}")
else:
    # Extract frames from videos using ffmpeg
    print("\n🎬 Extracting frames from videos...")
    for i, video in enumerate(inputs):
        src = os.path.join(CALIBRATION_INPUT, video)
        dst = os.path.join(RIG_IMAGES_DIR, f"cam{i+1:02d}")
        os.makedirs(dst, exist_ok=True)
        
        # Extract frames (1 frame per second, adjust as needed)
        cmd = f'ffmpeg -i "{src}" -vf "fps=1" "{dst}/frame_%04d.jpg"'
        subprocess.run(cmd, shell=True, check=True)
        frame_count = len(os.listdir(dst))
        print(f"   🎬 {video} → cam{i+1:02d} ({frame_count} frames)")

# ==============================================================================
# COLMAP RIG CALIBRATION PIPELINE
# ==============================================================================
COLMAP_DB = os.path.join(CALIB_WORKSPACE, "database.db")
COLMAP_SPARSE = os.path.join(CALIB_WORKSPACE, "sparse")
os.makedirs(COLMAP_SPARSE, exist_ok=True)

print("\n" + "=" * 40)
print("📸 COLMAP RIG CALIBRATION")
print("=" * 40)

# 1. Feature Extraction (one camera per folder)
print("\n1️⃣ Feature Extraction (single_camera_per_folder=1)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {RIG_IMAGES_DIR} \
    --ImageReader.single_camera_per_folder 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Sequential Matcher (for rig/video sequences)
print("\n2️⃣ Sequential Matcher...")
!colmap sequential_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper (with rig constraints)
print("\n3️⃣ Mapper (ba_refine_sensor_from_rig=0)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {RIG_IMAGES_DIR} \
    --output_path {COLMAP_SPARSE} \
    --Mapper.ba_refine_sensor_from_rig 0

# Find the model path (COLMAP creates subfolder like '0')
sparse_model = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model):
    sparse_model = COLMAP_SPARSE

# ==============================================================================
# 🔴 SANITY CHECK: RIG INTEGRITY
# ==============================================================================
print("\n" + "=" * 40)
print("🔍 SANITY CHECK: RIG INTEGRITY")
print("=" * 40)

# Convert to TXT for parsing
!colmap model_converter \
    --input_path {sparse_model} \
    --output_path {sparse_model} \
    --output_type TXT

# Check camera count
images_txt = os.path.join(sparse_model, "images.txt")
if os.path.exists(images_txt):
    with open(images_txt, 'r') as f:
        lines = [l for l in f.readlines() if not l.startswith('#') and l.strip()]
    # Each image takes 2 lines in COLMAP format
    num_cameras = len(lines) // 2
    print(f"   📷 Registered cameras: {num_cameras}")
    
    if num_cameras != 8:
        raise RuntimeError(f"🔴 CRITICAL ERROR: Expected 8 cameras, got {num_cameras}!")
    print("   ✅ All 8 cameras registered successfully")
else:
    raise FileNotFoundError("🔴 CRITICAL ERROR: images.txt not found!")

# Check reprojection error (parse from COLMAP output or run bundle_adjuster)
# For now, we'll trust the mapper output. Advanced: parse mapper logs.
print("   ✅ Rig calibration passed sanity checks")

# ==============================================================================
# SAVE CALIBRATED POSES
# ==============================================================================
print("\n📦 Saving calibrated poses...")

if os.path.exists(SPARSE_CALIBRATED):
    shutil.rmtree(SPARSE_CALIBRATED)
shutil.copytree(sparse_model, SPARSE_CALIBRATED)

# Also save as BIN
!colmap model_converter \
    --input_path {SPARSE_CALIBRATED} \
    --output_path {SPARSE_CALIBRATED} \
    --output_type BIN

print(f"\n✅ PHASE 1 COMPLETE: Perfect Poses Acquired!")
print(f"📍 Saved to: {SPARSE_CALIBRATED}")
print("=" * 60)


## ☁️ Cell 3.5: Phase 2 - Extraview Reconstruction (Dense Geometry)

**Goal:** Build dense point cloud from subject scene + extraviews.

**Input:** Main subject images + extra views (NOT calibration clip)

**Output:** `/content/output/sparse_geometric_source`


In [ ]:
# ============================================================
# CELL 3.5: PHASE 2 - EXTRAVIEW RECONSTRUCTION (DENSE GEOMETRY)
# ============================================================
# Goal: Build dense point cloud from subject scene + extraviews
# Input: Main subject images + extra views (NOT calibration clip)
# Output: /content/output/sparse_geometric_source
# ============================================================

import os
import shutil

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Path to SUBJECT images (main views + extraviews)
# This should contain: image00.jpg, image01.jpg, ... AND extra00.jpg, extra01.jpg, etc.
SUBJECT_IMAGES_INPUT = "/content/drive/MyDrive/4DGS_project/input/work_allviews/images"

# Output paths
EXTRAVIEW_WORKSPACE = "/content/colmap_extraview_workspace"
SPARSE_GEOMETRIC = "/content/output/sparse_geometric_source"
DENSE_OUTPUT = os.path.join(EXTRAVIEW_WORKSPACE, "dense")

print("=" * 60)
print("🎯 PHASE 2: EXTRAVIEW RECONSTRUCTION - DENSE GEOMETRY")
print("=" * 60)

# ==============================================================================
# DENSE RECONSTRUCTION SETTINGS
# ==============================================================================
PM_WINDOW_RADIUS = 5
PM_NUM_ITERATIONS = 5
PM_GEOM_CONSISTENCY = 1
SF_MIN_NUM_PIXELS = 5
SF_MAX_REPROJ_ERROR = 2
SF_MAX_DEPTH_ERROR = 0.01

# ==============================================================================
# PREPARE WORKSPACE
# ==============================================================================
print("\n📁 Preparing extraview workspace...")

if os.path.exists(EXTRAVIEW_WORKSPACE):
    shutil.rmtree(EXTRAVIEW_WORKSPACE)
os.makedirs(EXTRAVIEW_WORKSPACE, exist_ok=True)
os.makedirs(DENSE_OUTPUT, exist_ok=True)

COLMAP_DB = os.path.join(EXTRAVIEW_WORKSPACE, "database.db")
COLMAP_SPARSE = os.path.join(EXTRAVIEW_WORKSPACE, "sparse")
os.makedirs(COLMAP_SPARSE, exist_ok=True)

# ==============================================================================
# COLMAP RECONSTRUCTION (Fresh - NOT using calibrated poses yet)
# ==============================================================================
print("\n" + "=" * 40)
print("📸 COLMAP EXTRAVIEW RECONSTRUCTION")
print("=" * 40)

# 1. Feature Extraction
print("\n1️⃣ Feature Extraction (GPU)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Exhaustive Matcher (for maximum point coverage)
print("\n2️⃣ Exhaustive Matcher (GPU)...")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper
print("\n3️⃣ Mapper (Sparse Reconstruction)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --output_path {COLMAP_SPARSE}

# Find sparse model path
sparse_model = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model):
    sparse_model = COLMAP_SPARSE

# 4. Image Undistorter (Dense preparation)
print("\n4️⃣ Image Undistorter (Dense Prep)...")
!colmap image_undistorter \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --input_path {sparse_model} \
    --output_path {DENSE_OUTPUT} \
    --output_type COLMAP \
    --max_image_size 2000

# 5. Patch Match Stereo (Depth Maps)
print(f"\n5️⃣ Patch Match Stereo (GPU)...")
print(f"   ⚙️ Window={PM_WINDOW_RADIUS}, Iters={PM_NUM_ITERATIONS}, GeomCheck={PM_GEOM_CONSISTENCY}")
!colmap patch_match_stereo \
    --workspace_path {DENSE_OUTPUT} \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency {PM_GEOM_CONSISTENCY} \
    --PatchMatchStereo.window_radius {PM_WINDOW_RADIUS} \
    --PatchMatchStereo.num_iterations {PM_NUM_ITERATIONS} \
    --PatchMatchStereo.gpu_index 0

# 6. Stereo Fusion (Dense Point Cloud)
print(f"\n6️⃣ Stereo Fusion (Dense Cloud)...")
print(f"   ⚙️ MinPixels={SF_MIN_NUM_PIXELS}, MaxReproj={SF_MAX_REPROJ_ERROR}, MaxDepthErr={SF_MAX_DEPTH_ERROR}")
FUSED_PLY = os.path.join(DENSE_OUTPUT, "fused.ply")
!colmap stereo_fusion \
    --workspace_path {DENSE_OUTPUT} \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path {FUSED_PLY} \
    --StereoFusion.min_num_pixels {SF_MIN_NUM_PIXELS} \
    --StereoFusion.max_reproj_error {SF_MAX_REPROJ_ERROR} \
    --StereoFusion.max_depth_error {SF_MAX_DEPTH_ERROR}

# ==============================================================================
# SAVE GEOMETRIC SOURCE
# ==============================================================================
print("\n📦 Saving geometric source model...")

# Convert to TXT
!colmap model_converter \
    --input_path {sparse_model} \
    --output_path {sparse_model} \
    --output_type TXT

if os.path.exists(SPARSE_GEOMETRIC):
    shutil.rmtree(SPARSE_GEOMETRIC)
shutil.copytree(sparse_model, SPARSE_GEOMETRIC)

# Also copy dense PLY
if os.path.exists(FUSED_PLY):
    shutil.copy(FUSED_PLY, os.path.join(SPARSE_GEOMETRIC, "fused.ply"))
    ply_size = os.path.getsize(FUSED_PLY) / (1024 * 1024)
    print(f"   ☁️ Dense PLY: {ply_size:.2f} MB")

print(f"\n✅ PHASE 2 COMPLETE: Dense Geometry Acquired!")
print(f"📍 Sparse: {SPARSE_GEOMETRIC}")
print(f"☁️ Dense PLY: {FUSED_PLY}")
print("=" * 60)


## 🔀 Cell 3.6: Phase 3 - Alignment & Fusion (Final Model)

**Goal:** Align geometry to rig poses and create final training dataset.

**Input:**
- `sparse_calibrated` (perfect poses from rig)
- `sparse_geometric_source` (dense points from extraview)

**Output:** Final training-ready model


In [ ]:
# ============================================================
# CELL 3.6: PHASE 3 - ALIGNMENT & FUSION
# ============================================================
# Goal: Align "High Detail Geometry" to "Accurate Rig Poses"
# Input: 
#   - sparse_calibrated (perfect poses from rig)
#   - sparse_geometric_source (dense points from extraview)
# Output: Final training-ready model
# ============================================================

import os
import shutil

# ==============================================================================
# PATHS
# ==============================================================================
SPARSE_CALIBRATED = "/content/output/sparse_calibrated"  # Perfect Poses (from 3.2)
SPARSE_GEOMETRIC = "/content/output/sparse_geometric_source"  # Dense Points (from 3.5)
SPARSE_ALIGNED = "/content/output/sparse_aligned"  # Aligned geometry
FINAL_OUTPUT = "/content/data/my_scene/colmap_output/sparse/0"  # Training target

print("=" * 60)
print("🎯 PHASE 3: ALIGNMENT & FUSION")
print("=" * 60)

# ==============================================================================
# STEP 1: ALIGNMENT (Model Aligner)
# ==============================================================================
print("\n" + "=" * 40)
print("🔄 STEP 1: MODEL ALIGNMENT")
print("=" * 40)
print(f"   📥 Input (Geometry): {SPARSE_GEOMETRIC}")
print(f"   📐 Reference (Poses): {SPARSE_CALIBRATED}")

if os.path.exists(SPARSE_ALIGNED):
    shutil.rmtree(SPARSE_ALIGNED)
os.makedirs(SPARSE_ALIGNED, exist_ok=True)

# Run COLMAP model_aligner
# This aligns the geometric model to the rig coordinate system
!colmap model_aligner \
    --input_path {SPARSE_GEOMETRIC} \
    --ref_path {SPARSE_CALIBRATED} \
    --output_path {SPARSE_ALIGNED} \
    --ref_is_gps 0 \
    --alignment_type ecef

print(f"   ✅ Aligned model saved to: {SPARSE_ALIGNED}")

# ==============================================================================
# STEP 2: FUSION (Merge Poses + Geometry)
# ==============================================================================
print("\n" + "=" * 40)
print("🔀 STEP 2: FUSION (POSES + GEOMETRY)")
print("=" * 40)

# Prepare final output directory
if os.path.exists(FINAL_OUTPUT):
    shutil.rmtree(FINAL_OUTPUT)
os.makedirs(FINAL_OUTPUT, exist_ok=True)

# Step A: Copy POSES from sparse_calibrated (We trust Rig Poses)
print("\n   📋 Step A: Copying POSES from Rig Calibration...")
for pose_file in ["cameras.txt", "cameras.bin", "images.txt", "images.bin"]:
    src = os.path.join(SPARSE_CALIBRATED, pose_file)
    if os.path.exists(src):
        shutil.copy(src, FINAL_OUTPUT)
        print(f"      ✓ {pose_file}")

# Step B: Copy POINTS from sparse_aligned (We trust Extraview Geometry)
print("\n   📋 Step B: Copying POINTS from Aligned Geometry...")
for points_file in ["points3D.txt", "points3D.bin"]:
    src = os.path.join(SPARSE_ALIGNED, points_file)
    if os.path.exists(src):
        shutil.copy(src, FINAL_OUTPUT)
        print(f"      ✓ {points_file}")

# Also copy dense PLY if available
dense_ply_src = os.path.join(SPARSE_GEOMETRIC, "fused.ply")
dense_ply_dst = os.path.join(os.path.dirname(FINAL_OUTPUT), "dense_point_cloud.ply")
if os.path.exists(dense_ply_src):
    shutil.copy(dense_ply_src, dense_ply_dst)
    print(f"\n   ☁️ Dense PLY copied to: {dense_ply_dst}")

# ==============================================================================
# STEP 3: FILTER EXTRAVIEWS FROM FINAL MODEL
# ==============================================================================
print("\n" + "=" * 40)
print("🔍 STEP 3: FILTER EXTRAVIEWS")
print("=" * 40)

# We need to ensure only main cameras (no "extra") are in final images.txt
images_txt_path = os.path.join(FINAL_OUTPUT, "images.txt")

if os.path.exists(images_txt_path):
    # Backup original poses BEFORE filtering for sanity check
    pose_backup = {}
    
    with open(images_txt_path, 'r') as f:
        lines = f.readlines()
    
    # First pass: backup main camera poses
    print("\n   🔒 Backing up main camera poses...")
    data_lines = [l for l in lines if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(data_lines):
        if i + 1 >= len(data_lines):
            break
        metadata_line = data_lines[i]
        points_line = data_lines[i + 1]
        
        parts = metadata_line.strip().split()
        if len(parts) >= 10:
            img_name = parts[-1]
            if "extra" not in img_name.lower():
                # Store pose: QW, QX, QY, QZ, TX, TY, TZ
                pose = [float(parts[j]) for j in range(1, 8)]
                pose_backup[img_name] = pose
                print(f"      ✓ Backed up: {img_name}")
        i += 2
    
    # Second pass: filter out extraviews
    print("\n   🗑️ Filtering extraview images...")
    count_kept = 0
    count_deleted = 0
    
    with open(images_txt_path, 'w') as f_out:
        # Write headers
        for line in lines:
            if line.startswith('#'):
                f_out.write(line)
        
        # Process data lines
        i = 0
        while i < len(data_lines):
            if i + 1 >= len(data_lines):
                break
            metadata_line = data_lines[i]
            points_line = data_lines[i + 1]
            
            parts = metadata_line.strip().split()
            if len(parts) >= 10:
                img_name = parts[-1]
                if "extra" not in img_name.lower():
                    f_out.write(metadata_line)
                    f_out.write(points_line)
                    count_kept += 1
                else:
                    count_deleted += 1
            i += 2
    
    print(f"      ✅ Kept: {count_kept} main cameras")
    print(f"      🗑️ Removed: {count_deleted} extraviews")
    
    # Third pass: verify poses haven't drifted (sanity check)
    print("\n   🔍 Verifying pose integrity...")
    with open(images_txt_path, 'r') as f:
        lines = f.readlines()
    
    data_lines = [l for l in lines if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(data_lines):
        if i + 1 >= len(data_lines):
            break
        metadata_line = data_lines[i]
        parts = metadata_line.strip().split()
        if len(parts) >= 10:
            img_name = parts[-1]
            if img_name in pose_backup:
                new_pose = [float(parts[j]) for j in range(1, 8)]
                old_pose = pose_backup[img_name]
                
                for j, (old_val, new_val) in enumerate(zip(old_pose, new_pose)):
                    diff = abs(old_val - new_val)
                    if diff > 1e-6:
                        raise RuntimeError(
                            f"🔴 CRITICAL ERROR: Main camera poses have drifted!\n"
                            f"   Image: {img_name}\n"
                            f"   Parameter {j}: {old_val} → {new_val} (diff: {diff})"
                        )
        i += 2
    
    print("      ✅ Pose integrity verified - no drift detected")

# ==============================================================================
# STEP 4: CONVERT TO BIN FORMAT
# ==============================================================================
print("\n   🔄 Converting to BIN format...")
!colmap model_converter \
    --input_path {FINAL_OUTPUT} \
    --output_path {FINAL_OUTPUT} \
    --output_type BIN

# ==============================================================================
# FINAL SANITY CHECKS
# ==============================================================================
print("\n" + "=" * 40)
print("✅ FINAL SANITY CHECKS")
print("=" * 40)

# Check all required files exist
required_files = ["cameras.bin", "images.bin", "points3D.bin"]
missing = [f for f in required_files if not os.path.exists(os.path.join(FINAL_OUTPUT, f))]

if missing:
    raise RuntimeError(f"🔴 CRITICAL ERROR: Missing files: {missing}")

print("   ✅ All required .bin files present")

# Check points3D is not empty
points_size = os.path.getsize(os.path.join(FINAL_OUTPUT, "points3D.bin"))
print(f"   📊 points3D.bin size: {points_size / 1024:.2f} KB")

if points_size < 1000:
    print("   ⚠️ WARNING: points3D.bin seems small - may have few points")

# Check images count
images_bin = os.path.join(FINAL_OUTPUT, "images.bin")
images_size = os.path.getsize(images_bin)
print(f"   📊 images.bin size: {images_size / 1024:.2f} KB")

print("\n" + "=" * 60)
print("🎉 PHASE 3 COMPLETE: ALIGNMENT & FUSION SUCCESSFUL!")
print("=" * 60)
print(f"\n📍 Final Model: {FINAL_OUTPUT}")
print("   Contains:")
print("   ├── cameras.bin  (from Rig Calibration)")
print("   ├── images.bin   (from Rig Calibration, filtered)")
print("   └── points3D.bin (from Extraview, aligned)")
print("\n👉 Ready for Training!")


## ✅ Cell 4: Pipeline Sanity Checks

Verify masks and training paths before starting training.


In [ ]:
# ============================================================
# CELL 4: PIPELINE SANITY CHECKS
# ============================================================

import os
import cv2
import numpy as np

# ==============================================================================
# MASK VERIFICATION
# ==============================================================================
MASK_PATH = "/content/data/my_scene/masks"  # Adjust as needed

print("=" * 60)
print("🔍 PIPELINE SANITY CHECKS")
print("=" * 60)

print("\n📋 Checking Masks...")

if not os.path.exists(MASK_PATH):
    raise FileNotFoundError(f"🔴 ERROR: Mask directory not found: {MASK_PATH}")

mask_files = [f for f in os.listdir(MASK_PATH) if f.endswith(('.png', '.jpg'))]

if len(mask_files) == 0:
    raise RuntimeError(f"🔴 ERROR: No mask files found in {MASK_PATH}")

print(f"   📁 Found {len(mask_files)} mask files")

# Check for empty/all-black masks
empty_masks = []
for mask_file in mask_files:
    mask_path = os.path.join(MASK_PATH, mask_file)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        empty_masks.append(mask_file)
        continue
    
    non_zero = np.count_nonzero(mask)
    if non_zero == 0:
        empty_masks.append(mask_file)

if empty_masks:
    print(f"   ⚠️ WARNING: {len(empty_masks)} empty/black masks detected:")
    for m in empty_masks[:5]:
        print(f"      - {m}")
    if len(empty_masks) > 5:
        print(f"      ... and {len(empty_masks) - 5} more")
else:
    print("   ✅ All masks have non-zero content")

# ==============================================================================
# TRAINING PATH VERIFICATION
# ==============================================================================
print("\n📋 Checking Training Paths...")

TRAINING_DATA = "/content/data/my_scene"
SPARSE_MODEL = os.path.join(TRAINING_DATA, "colmap_output/sparse/0")

# Check sparse model
if not os.path.exists(SPARSE_MODEL):
    raise FileNotFoundError(f"🔴 ERROR: Sparse model not found: {SPARSE_MODEL}")

required = ["cameras.bin", "images.bin", "points3D.bin"]
for f in required:
    if not os.path.exists(os.path.join(SPARSE_MODEL, f)):
        raise FileNotFoundError(f"🔴 ERROR: Missing {f} in sparse model")

print(f"   ✅ Sparse model verified: {SPARSE_MODEL}")

# Check images directory
IMAGES_DIR = os.path.join(TRAINING_DATA, "images")
if os.path.exists(IMAGES_DIR):
    img_count = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png'))])
    print(f"   ✅ Images directory: {img_count} images")
else:
    print(f"   ⚠️ WARNING: Images directory not found at expected path")

print("\n" + "=" * 60)
print("✅ ALL SANITY CHECKS PASSED - READY FOR TRAINING!")
print("=" * 60)


---

## 📝 Remaining Pipeline Steps

The following cells contain mask generation, training, and rendering steps.
These are copied from the golden source notebook.


## 🎭 Mask Generation & Training

**TODO:** Add mask generation, training, and rendering cells here.

These cells should include:
- SAM2 mask generation
- 4DGS training
- Rendering and visualization
